# MPP state-budget evidence review

## tl;dr

The fixed search expands from six to ten valid candidates per shape, with 216 complete outputs checked. Search minima are not accepted performance wins; all eight fresh selections were slower than their selected trial. This companion audits saved evidence, and does not launch GPU work.

## Context & Methods

Technical companion to the existing Sphinx report. Four FP32 GEMM shapes, twelve candidate configurations per old/new compiler, five timing samples and preallocated outputs. GPU measurements are no-counter command-buffer intervals, not isolated kernel timestamps.

### Key Assumptions

Saved native/Torch/MPS numerical receipts are checked against the exact recorded output size. This audit recomputes raw timing denominators and selections but cannot reconstruct an idle desktop or prove the source of variable load. See `protocol.md`, `notes.md`, and the recorded binary/source hashes.

## Data

### 1. Load the saved search and independent auditor

Run from this notebook's directory. Only the registered experiment subtree is read.

In [1]:
import importlib.util
import json
from pathlib import Path

root = Path.cwd()
spec = importlib.util.spec_from_file_location('state_budget_audit', root / 'audit.py')
auditor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(auditor)
report = json.loads((root / 'search/results.json').read_text())
checked = auditor.audit_search(report, root / 'search')
print({key: value for key, value in checked.items() if key != 'summary'})

{'passed': True, 'use': 'candidate admission and numerical validation; uncontrolled search timings are not accepted performance gains', 'attempted_candidates': 96, 'valid_candidates': 64, 'complete_outputs': 216, 'checked_elements': 1925296182, 'unchanged_common_candidates': 24}


## Results

### 2. Admission and fresh-measurement instability

Exact lookup by shape and compiler variant. The last number is fresh GPU time divided by the same selected configuration's search time, not a compiler speedup.

In [2]:
for row in checked['summary']:
    print('x'.join(map(str, row['shape'])), row['variant'],
          f"valid={row['valid_candidates']}/12",
          f"fresh/search={row['selection_to_fresh_gpu_ratio']:.3f}")
assert checked['unchanged_common_candidates'] == 24
assert checked['complete_outputs'] == 216
assert all(row['selection_to_fresh_gpu_ratio'] > 1 for row in checked['summary'])

1024x1024x1024 reference valid=6/12 fresh/search=1.878
1024x1024x1024 candidate valid=10/12 fresh/search=1.373
4096x4096x4096 candidate valid=10/12 fresh/search=1.471
4096x4096x4096 reference valid=6/12 fresh/search=1.225
1025x1025x1024 reference valid=6/12 fresh/search=1.413
1025x1025x1024 candidate valid=10/12 fresh/search=1.268
4096x4096x11008 candidate valid=10/12 fresh/search=1.243
4096x4096x11008 reference valid=6/12 fresh/search=1.351


### 3. Current numerical gates and old-library negative controls

A test exits successfully only when nonzero assertions run; expected old-library failures are separately labeled.

In [3]:
receipt = json.loads((root / 'final-correctness/results.json').read_text())
assert receipt['passed'] and receipt['metadata']['artifacts_unchanged']
assert all(row['exit_code'] == 0 for row in receipt['metadata']['builds'])
positive = [row for row in receipt['results'] if not row['expected_failure']]
negative = [row for row in receipt['results'] if row['expected_failure']]
assert all(row['passed'] and row['passed_assertions'] > 0 for row in positive)
assert len(negative) == 2 and all(row['passed'] and row['exit_code'] != 0 for row in negative)
print('Passing numerical test invocations:', len(positive))
print('Expected old-library negative controls:', len(negative))
print('Passing assertions:', sum(row['passed_assertions'] for row in positive))

Passing numerical test invocations: 12
Expected old-library negative controls: 2
Passing assertions: 915242


## Takeaways

Admission and numerical correctness are supported by saved, source-backed evidence. The 24 common candidate sources are unchanged, so their timing differences are not compiler transformations. Timing/model selection still requires a quiet-window, held-out and counterbalanced replay; no default schedule or cost coefficient is promoted. The PyTorch/MPS-equivalent performance objective remains open.